# 04 — Panel Scoring (Pre / Post Windows)

Scores each panel user's text in the **pre-baseline** and **post-outcome** windows,
then merges with exposure labels to build the analysis panel.

**Pre-period definition (Sep–Nov, before first anchor comment):**
- Exposed users: all Sep–Nov posts/comments *before* their first anchor comment
- Unexposed users: all Sep–Nov posts/comments (no anchor comment, so full Sep–Nov)
- This replaces the fixed August window to maximise coverage (~36% vs ~7%)

**Post-period:** December–May of each cycle year

**Inputs:**
- `cleaned_output/r_gradadmissions_posts.cleaned.jsonl` + `Grad Admissions Comments.jsonl`
- `data/processed_v2/exposure_labels_v2.parquet` (from notebook 03)
- `data/processed_v2/anchor_posts_v2.parquet` (from notebook 03)

**Outputs:**
- `data/processed_v2/panel_scores_v2.parquet`
- `data/processed_v2/post_level_scores_v2.parquet`
- `data/processed_v2/dose_exposure_v2.parquet`


In [1]:
import json
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timezone
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_V2   = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'

POSTS_CLEAN    = ROOT / 'cleaned_output' / 'r_gradadmissions_posts.cleaned.jsonl'
COMMENTS_RAW   = ROOT / 'Grad Admissions Comments.jsonl'
EXPOSURE_PATH  = DATA_V2 / 'exposure_labels_v2.parquet'
ANCHOR_PATH    = DATA_V2 / 'anchor_posts_v2.parquet'
OUT_PATH       = DATA_V2 / 'panel_scores_v2.parquet'

CYCLES = {
    1: {
        'anchor_start': datetime(2023,  9,  1, tzinfo=timezone.utc),
        'anchor_end':   datetime(2023, 11, 30, 23, 59, 59, tzinfo=timezone.utc),
        'post_start':   datetime(2023, 12,  1, tzinfo=timezone.utc),
        'post_end':     datetime(2024,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
    2: {
        'anchor_start': datetime(2024,  9,  1, tzinfo=timezone.utc),
        'anchor_end':   datetime(2024, 11, 30, 23, 59, 59, tzinfo=timezone.utc),
        'post_start':   datetime(2024, 12,  1, tzinfo=timezone.utc),
        'post_end':     datetime(2025,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
}

print('Paths OK:', all(p.exists() for p in [POSTS_CLEAN, COMMENTS_RAW, EXPOSURE_PATH, ANCHOR_PATH]))


Paths OK: True


## 1) Load panel users

In [2]:
exposure = pd.read_parquet(EXPOSURE_PATH)
print(f'Panel users: {exposure["author"].nunique():,}  |  rows: {len(exposure):,}')
print(exposure['exposed'].value_counts())
panel_users = set(exposure['author'])

anchor_posts_df = pd.read_parquet(ANCHOR_PATH, columns=['id', 'cycle'])
anchor_ids = set(anchor_posts_df['id'].astype(str))
print(f'Anchor post IDs: {len(anchor_ids):,}')


Panel users: 22,518  |  rows: 23,392
exposed
False    20521
True      2871
Name: count, dtype: int64
Anchor post IDs: 1,024


## 2) Find first anchor comment per exposed user


In [3]:
# Scan raw comments to find each exposed user's first anchor comment timestamp
# Pre-period cutoff = this timestamp; activity before it counts as pre-period
print('Scanning comments for first anchor comment per exposed user...')
first_anchor_comment = {}  # author -> datetime

with open(COMMENTS_RAW) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        r = json.loads(line)
        author  = r.get('author')
        post_id = r.get('link_id', '').replace('t3_', '')
        if author not in panel_users or post_id not in anchor_ids: continue
        dt = datetime.fromtimestamp(r['created_utc'], tz=timezone.utc)
        if author not in first_anchor_comment or dt < first_anchor_comment[author]:
            first_anchor_comment[author] = dt

print(f'First anchor comment found for {len(first_anchor_comment):,} exposed users')


Scanning comments for first anchor comment per exposed user...


First anchor comment found for 3,264 exposed users


In [4]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score_texts_all(texts):
    if not texts:
        return np.array([]), np.array([]), np.array([]), np.array([])
    anx  = sigmoid(clf_anx.decision_function(texts))
    dep  = sigmoid(clf_dep.decision_function(texts))
    str_ = sigmoid(clf_str.decision_function(texts))
    mean = np.stack([anx, dep, str_], axis=1).mean(axis=1)
    return anx, dep, str_, mean


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Tr

Classifiers loaded.


## 4) Build corpus and score


In [5]:
def assign_window(author, dt, cycle):
    """
    Pre:  Sep-Nov of the cycle AND before first anchor comment (exposed)
          or full Sep-Nov (unexposed)
    Post: Dec-May of the cycle
    """
    w = CYCLES[cycle]
    if w['post_start'] <= dt <= w['post_end']:
        return 'post'
    if w['anchor_start'] <= dt <= w['anchor_end']:
        cutoff = first_anchor_comment.get(author)  # None if unexposed
        if cutoff is None or dt < cutoff:
            return 'pre'
    return None

user_cycles = exposure.groupby('author')['cycle'].apply(list).to_dict()

records = []

def process_file(path, text_field):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            author = r.get('author')
            if author not in panel_users: continue
            dt = datetime.fromtimestamp(r['created_utc'], tz=timezone.utc)
            text = r.get(text_field, '') or ''
            for cycle in user_cycles.get(author, []):
                window = assign_window(author, dt, cycle)
                if window:
                    records.append({
                        'author': author, 'cycle': cycle, 'window': window,
                        'created_dt': dt.isoformat(), 'clean_text': text
                    })

print('Processing posts...')
process_file(POSTS_CLEAN, 'clean_text')
print('Processing comments...')
process_file(COMMENTS_RAW, 'body')

corpus = pd.DataFrame(records)
print(f'\nWindow distribution:')
print(corpus.groupby(['cycle', 'window']).size())
print(f'Unique authors: {corpus["author"].nunique():,}')


Processing posts...


Processing comments...



Window distribution:
cycle  window
1      post      68786
       pre       24816
2      post      72458
       pre       27525
dtype: int64
Unique authors: 21,110


## 4a) Score all records


In [6]:
texts = corpus['clean_text'].tolist()
print(f'Scoring {len(texts):,} records...')
corpus['anx_score'], corpus['dep_score'], corpus['str_score'], corpus['mean_mh_score'] = score_texts_all(texts)
print('Done.')
corpus[['anx_score', 'dep_score', 'str_score', 'mean_mh_score']].describe().round(4)

Scoring 193,585 records...


Done.


,anx_score,dep_score,str_score,mean_mh_score
count,193585.0000,193585.0000,193585.0000,193585.0000
mean,0.4045,0.4349,0.4688,0.4361
std,0.1052,0.1191,0.0982,0.0851
min,0.0873,0.0532,0.1124,0.0843
25%,0.3494,0.3575,0.4031,0.3812
50%,0.4095,0.4221,0.4627,0.4359
75%,0.4633,0.4872,0.5223,0.4925
max,0.9881,0.8664,0.9398,0.8302


## 4b) Save post-level scores (for post-level DiD in NB06)

In [7]:
# Save individual post/comment records with scores before aggregation.
# Used by NB06 for post-level DiD (recovers ~147K observations vs 1,094 user-means).
POST_LEVEL_PATH = DATA_V2 / 'post_level_scores_v2.parquet'

post_level = corpus[['author', 'cycle', 'window', 'created_dt',
                      'anx_score', 'dep_score', 'str_score', 'mean_mh_score']].copy()
post_level['created_dt'] = pd.to_datetime(post_level['created_dt'], utc=True)

post_level.to_parquet(POST_LEVEL_PATH, index=False)
print(f'Saved {len(post_level):,} post-level rows → {POST_LEVEL_PATH}')
print(post_level.groupby(['cycle', 'window']).size())

Saved 193,585 post-level rows → /Users/veda/Documents/reddit-gradadmissions-distress/data/processed_v2/post_level_scores_v2.parquet
cycle  window
1      post      68786
       pre       24816
2      post      72458
       pre       27525
dtype: int64


## 4c) Compute dose-response data (# anchor-thread comments per user)

In [8]:
import json as _json

ANCHOR_PATH = DATA_V2 / 'anchor_posts_v2.parquet'
DOSE_PATH   = DATA_V2 / 'dose_exposure_v2.parquet'

anchor_posts_df = pd.read_parquet(ANCHOR_PATH, columns=['id', 'cycle'])
anchor_ids      = set(anchor_posts_df['id'].astype(str))

# Load raw comments for dose calculation (need link_id)
dose_rows = []
with open(COMMENTS_RAW) as f:
    for line in f:
        r = _json.loads(line)
        post_id = r.get('link_id', '').replace('t3_', '')
        author  = r.get('author', '')
        if post_id in anchor_ids and author in panel_users:
            dose_rows.append({'author': author, 'post_id': post_id})

dose_comments = pd.DataFrame(dose_rows)

pid_to_cycle = anchor_posts_df.set_index('id')['cycle'].to_dict()
dose_comments['cycle'] = dose_comments['post_id'].map(pid_to_cycle)

dose = (
    dose_comments.groupby(['author', 'cycle'])
    .size()
    .reset_index(name='n_anchor_comments')
)
dose['log1p_n_anchor'] = np.log1p(dose['n_anchor_comments'])

dose.to_parquet(DOSE_PATH, index=False)
print(f'Saved {len(dose):,} dose records → {DOSE_PATH}')
print(dose['n_anchor_comments'].describe().round(2))


Saved 3,341 dose records → /Users/veda/Documents/reddit-gradadmissions-distress/data/processed_v2/dose_exposure_v2.parquet
count    3341.00
mean        2.32
std         4.19
min         1.00
25%         1.00
50%         1.00
75%         2.00
max       122.00
Name: n_anchor_comments, dtype: float64


## 5) Aggregate per (author, cycle, window)

In [9]:
agg = (
    corpus
    .groupby(['author', 'cycle', 'window'])
    .agg(
        mh_score  = ('mean_mh_score', 'mean'),
        anx_score = ('anx_score',     'mean'),
        dep_score = ('dep_score',     'mean'),
        str_score = ('str_score',     'mean'),
        n_posts   = ('mean_mh_score', 'count'),
    )
    .reset_index()
)

def pivot_window(window_label, prefix):
    sub = agg[agg['window'] == window_label].drop(columns='window')
    return sub.rename(columns={
        'mh_score':  f'{prefix}_mh_score',
        'anx_score': f'{prefix}_anx_score',
        'dep_score': f'{prefix}_dep_score',
        'str_score': f'{prefix}_str_score',
        'n_posts':   f'{prefix}_n_posts',
    })

pre  = pivot_window('pre',  'pre')
post = pivot_window('post', 'post')

scores = pre.merge(post, on=['author', 'cycle'], how='inner')
print(f'Users with both pre and post observations: {len(scores):,}')

Users with both pre and post observations: 8,124


## 6) Merge with exposure labels

In [10]:
panel = exposure.merge(scores, on=['author', 'cycle'], how='inner')
print(f'Final panel rows: {len(panel):,}')
print(f'Unique users:     {panel["author"].nunique():,}')
print(f'\nCoverage: {100 * panel["author"].nunique() / len(panel_users):.1f}% of panel users have pre+post scores')
print('\nExposure breakdown:')
print(panel.groupby(['cycle', 'exposed']).size())

Final panel rows: 8,124
Unique users:     7,921

Coverage: 35.2% of panel users have pre+post scores

Exposure breakdown:
cycle  exposed
1      False      3420
       True        451
2      False      3737
       True        516
dtype: int64


In [11]:
# Score distribution check
print('Pre-period MH scores:')
print(panel.groupby('exposed')['pre_mh_score'].describe().round(4))
print('\nPost-period MH scores:')
print(panel.groupby('exposed')['post_mh_score'].describe().round(4))

Pre-period MH scores:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    7157.0  0.4037  0.0699  0.1309  0.3616  0.4055  0.4453  0.7498
True      967.0  0.4158  0.0547  0.1989  0.3829  0.4157  0.4481  0.6034

Post-period MH scores:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    7157.0  0.4271  0.0579  0.1519  0.3974  0.4305  0.4593  0.7784
True      967.0  0.4352  0.0461  0.2357  0.4102  0.4351  0.4584  0.6676


## 7) Save

> **Approval gate:** Review coverage stats and score distributions above before running this cell.

In [12]:
out_cols = [
    'author', 'cycle', 'exposed', 'exposure_intensity',
    'pre_mh_score',  'pre_anx_score',  'pre_dep_score',  'pre_str_score',  'pre_n_posts',
    'post_mh_score', 'post_anx_score', 'post_dep_score', 'post_str_score', 'post_n_posts',
]
panel[out_cols].to_parquet(OUT_PATH, index=False)
print(f'Saved {len(panel):,} rows → {OUT_PATH}')
print('Columns:', out_cols)
print('\nexposure_intensity distribution:')
print(panel.groupby(['cycle','exposure_intensity']).size().unstack(fill_value=0))


Saved 8,124 rows → /Users/veda/Documents/reddit-gradadmissions-distress/data/processed_v2/panel_scores_v2.parquet
Columns: ['author', 'cycle', 'exposed', 'exposure_intensity', 'pre_mh_score', 'pre_anx_score', 'pre_dep_score', 'pre_str_score', 'pre_n_posts', 'post_mh_score', 'post_anx_score', 'post_dep_score', 'post_str_score', 'post_n_posts']

exposure_intensity distribution:
exposure_intensity     0   1   2    3
cycle                                
1                   3420  89  62  300
2                   3737  83  92  341
